# Phase 8 — Unity Catalog

**Objectifs:**
- [x] Catalog
- [x] Schemas
- [x] Tables
- [x] Permissions
- [x] Data Lineage

**Livrable:** Data Governance

**Unity Catalog (UC)** = la couche de **gouvernance** de Databricks: un seul endroit pour organiser, documenter, sécuriser et tracer toutes les données.

```
Metastore
└── Catalog: workspace
    ├── Schema: raw     → Volume landing (fichiers bruts)
    ├── Schema: bronze  → yellow_trips, taxi_zones
    ├── Schema: silver  → yellow_trips, yellow_trips_rejected, taxi_zones
    ├── Schema: gold    → fact_trips, dim_zone, dim_date, kpi_daily, kpi_pickup_zone
    └── Schema: ops     → data_dictionary, ingestion_log, dq_results
```
Nom complet d'une table: **`catalog.schema.table`** (namespace à 3 niveaux).

## 0. Setup

In [1]:
from lakehouse_config import *
spark = get_spark()

from pyspark.sql import functions as F
import pandas as pd

## 1. Catalog

In [2]:
spark.sql("SHOW CATALOGS").show()
spark.sql(f"DESCRIBE CATALOG EXTENDED {CATALOG}").show(truncate=False)

+---------+
|  catalog|
+---------+
|  samples|
|   system|
|workspace|
+---------+



+-----------------------+---------------------------------------------------------+
|info_name              |info_value                                               |
+-----------------------+---------------------------------------------------------+
|Catalog Name           |workspace                                                |
|Comment                |                                                         |
|Owner                  |_workspace_admins_workspace_7474648698937928             |
|Catalog Type           |Regular                                                  |
|Created By             |64427f40-d6c3-40e7-91e8-20d10bb93067                     |
|Created At             |2026-11-06 AD at 19:56:56 UTC                            |
|Updated By             |64427f40-d6c3-40e7-91e8-20d10bb93067                     |
|Updated At             |2026-11-06 AD at 19:56:58 UTC                            |
|Storage Root           |                                                   

## 2. Schemas
Chaque schéma correspond à une **couche** de l'architecture Medallion. On les **documente** avec un commentaire.

In [3]:
schema_comments = {
    SCHEMA_RAW: "Zone d atterrissage: fichiers bruts (Volume landing)",
    SCHEMA_BRONZE: "Bronze: donnees brutes en Delta + metadata d ingestion",
    SCHEMA_SILVER: "Silver: donnees nettoyees, typees, dedoublonnees + rejets",
    SCHEMA_GOLD: "Gold: modele en etoile et KPIs metier",
    SCHEMA_OPS: "Operations: data dictionary, journal d ingestion, resultats data quality",
}
for schema, comment in schema_comments.items():
    spark.sql(f"COMMENT ON SCHEMA {schema} IS '{comment}'")

spark.sql(f"""
    SELECT schema_name, comment, created
    FROM {CATALOG}.information_schema.schemata
    WHERE schema_name IN ('raw', 'bronze', 'silver', 'gold', 'ops')
    ORDER BY created
""").show(truncate=False)

+-----------+------------------------------------------------------------------------+-----------------------+
|schema_name|comment                                                                 |created                |
+-----------+------------------------------------------------------------------------+-----------------------+
|raw        |Zone d atterrissage: fichiers bruts (Volume landing)                    |2026-11-07 16:45:07.459|
|bronze     |Bronze: donnees brutes en Delta + metadata d ingestion                  |2026-11-07 16:45:08.593|
|silver     |Silver: donnees nettoyees, typees, dedoublonnees + rejets               |2026-11-07 16:45:09.148|
|gold       |Gold: modele en etoile et KPIs metier                                   |2026-11-07 16:45:09.71 |
|ops        |Operations: data dictionary, journal d ingestion, resultats data quality|2026-11-07 16:45:10.218|
+-----------+------------------------------------------------------------------------+-----------------------+



In [4]:
spark.sql(f"SHOW VOLUMES IN {SCHEMA_RAW}").show()

+--------+-----------+
|database|volume_name|
+--------+-----------+
|     raw|    landing|
+--------+-----------+



## 3. Tables
### 3.1 Inventaire (via `information_schema`)

In [5]:
inventory = spark.sql(f"""
    SELECT table_schema AS schema, table_name, table_type, data_source_format AS format, comment
    FROM {CATALOG}.information_schema.tables
    WHERE table_schema IN ('bronze', 'silver', 'gold', 'ops')
    ORDER BY CASE table_schema WHEN 'bronze' THEN 1 WHEN 'silver' THEN 2 WHEN 'gold' THEN 3 ELSE 4 END, table_name
""").toPandas()
inventory["rows"] = [spark.table(f"{CATALOG}.{s}.{t}").count() for s, t in zip(inventory.schema, inventory.table_name)]
inventory

,schema,table_name,table_type,format,comment,rows
0,bronze,taxi_zones,MANAGED,DELTA,Bronze - Referentiel des zones TLC brut + meta...,265
1,bronze,yellow_trips,MANAGED,DELTA,Bronze - NYC Yellow Taxi trips bruts (TLC) + m...,5972150
2,silver,taxi_zones,MANAGED,DELTA,Silver - zones TLC normalisees,265
3,silver,yellow_trips,MANAGED,DELTA,"Silver - courses valides, typees, dedoublonnee...",5771620
4,silver,yellow_trips_rejected,MANAGED,DELTA,Silver - courses rejetees avec motif (reject_r...,200529
5,gold,dim_date,MANAGED,DELTA,None,60
6,gold,dim_zone,MANAGED,DELTA,None,265
7,gold,fact_trips,MANAGED,DELTA,None,5771620
8,gold,kpi_daily,MANAGED,DELTA,None,60
9,gold,kpi_pickup_zone,MANAGED,DELTA,None,260


### 3.2 Documentation des tables et colonnes Gold

In [6]:
table_comments = {
    T_GOLD_FACT_TRIPS: "Gold - Fait: une ligne par course valide (mesures + cles vers dim_date / dim_zone)",
    T_GOLD_DIM_ZONE: "Gold - Dimension zones TLC (pickup et dropoff)",
    T_GOLD_DIM_DATE: "Gold - Dimension calendrier",
    T_GOLD_DAILY_KPIS: "Gold - KPIs journaliers (courses, CA, panier moyen, pourboire)",
    T_GOLD_ZONE_KPIS: "Gold - KPIs par zone de prise en charge",
    T_DATA_DICTIONARY: "Ops - Data dictionary (Phase 1)",
    T_INGESTION_LOG: "Ops - Journal des fichiers ingeres (Phase 3 et 7)",
    T_DQ_RESULTS: "Ops - Historique des controles data quality (Phase 6)",
}
for t, c in table_comments.items():
    spark.sql(f"COMMENT ON TABLE {t} IS '{c}'")

column_comments = {
    "trip_id": "Cle technique SHA-256 de la course",
    "date_key": "Cle vers dim_date (yyyyMMdd)",
    "pu_zone_key": "Zone de prise en charge -> dim_zone.zone_key",
    "do_zone_key": "Zone de depose -> dim_zone.zone_key",
    "total_amount": "Montant total facture (USD, hors pourboire especes)",
    "trip_duration_min": "Duree de la course en minutes",
}
for col, c in column_comments.items():
    spark.sql(f"ALTER TABLE {T_GOLD_FACT_TRIPS} ALTER COLUMN {col} COMMENT '{c}'")

spark.sql(f"DESCRIBE TABLE {T_GOLD_FACT_TRIPS}").filter("comment IS NOT NULL").show(truncate=False)

+------------------------+-------------+---------------------------------------------------+
|col_name                |data_type    |comment                                            |
+------------------------+-------------+---------------------------------------------------+
|trip_id                 |string       |Cle technique SHA-256 de la course                 |
|date_key                |int          |Cle vers dim_date (yyyyMMdd)                       |
|pu_zone_key             |int          |Zone de prise en charge -> dim_zone.zone_key       |
|do_zone_key             |int          |Zone de depose -> dim_zone.zone_key                |
|trip_duration_min       |double       |Duree de la course en minutes                      |
|total_amount            |decimal(10,2)|Montant total facture (USD, hors pourboire especes)|
|# Clustering Information|             |                                                   |
|# col_name              |data_type    |comment                       

### 3.3 Tags (classification)
Les **tags** permettent de classer les tables (couche, domaine, sensibilité) et de les retrouver par recherche.

In [7]:
layer_of = {"bronze": "bronze", "silver": "silver", "gold": "gold", "ops": "ops"}
for s, t in zip(inventory.schema, inventory.table_name):
    try:
        spark.sql(f"ALTER TABLE {CATALOG}.{s}.{t} SET TAGS ('layer' = '{layer_of[s]}', 'domain' = 'nyc_taxi', 'pii' = 'false')")
    except Exception as e:
        print(f"{s}.{t}: tags non appliques -> {str(e).splitlines()[0][:120]}")

spark.sql(f"""
    SELECT schema_name, table_name, tag_name, tag_value
    FROM {CATALOG}.information_schema.table_tags
    WHERE schema_name = 'gold' ORDER BY table_name, tag_name
""").show(truncate=False)

+-----------+---------------+--------+---------+
|schema_name|table_name     |tag_name|tag_value|
+-----------+---------------+--------+---------+
|gold       |dim_date       |domain  |nyc_taxi |
|gold       |dim_date       |layer   |gold     |
|gold       |dim_date       |pii     |false    |
|gold       |dim_zone       |domain  |nyc_taxi |
|gold       |dim_zone       |layer   |gold     |
|gold       |dim_zone       |pii     |false    |
|gold       |fact_trips     |domain  |nyc_taxi |
|gold       |fact_trips     |layer   |gold     |
|gold       |fact_trips     |pii     |false    |
|gold       |kpi_daily      |domain  |nyc_taxi |
|gold       |kpi_daily      |layer   |gold     |
|gold       |kpi_daily      |pii     |false    |
|gold       |kpi_pickup_zone|domain  |nyc_taxi |
|gold       |kpi_pickup_zone|layer   |gold     |
|gold       |kpi_pickup_zone|pii     |false    |
+-----------+---------------+--------+---------+



## 4. Permissions

UC gère les droits avec `GRANT` / `REVOKE` sur des **objets** (catalog, schema, table, volume) pour des **principals** (utilisateurs, groupes, service principals). Les droits sont **hérités**: un droit sur un schéma s'applique à toutes ses tables.

| Privilège | Permet |
|---|---|
| `USE CATALOG` / `USE SCHEMA` | Accéder au conteneur (prérequis) |
| `SELECT` | Lire une table / vue |
| `MODIFY` | Insérer, mettre à jour, supprimer |
| `CREATE TABLE` | Créer des tables dans un schéma |
| `READ VOLUME` / `WRITE VOLUME` | Lire / écrire des fichiers d'un volume |
| `ALL PRIVILEGES` | Tous les droits |

### 4.1 Droits actuels

In [8]:
spark.sql("SELECT current_user() AS moi").show(truncate=False)
spark.sql(f"SHOW GRANTS ON SCHEMA {SCHEMA_GOLD}").show(truncate=False)

+--------------------------+
|moi                       |
+--------------------------+
|marwane2001morfi@gmail.com|
+--------------------------+



+---------+----------+----------+---------+
|Principal|ActionType|ObjectType|ObjectKey|
+---------+----------+----------+---------+
+---------+----------+----------+---------+



### 4.2 Modèle de droits proposé (principe du moindre privilège)

| Groupe | raw | bronze | silver | gold | ops |
|---|---|---|---|---|---|
| `data_engineers` | READ/WRITE VOLUME | ALL | ALL | ALL | ALL |
| `data_analysts` | — | — | SELECT | SELECT | — |
| `bi_users` | — | — | — | SELECT | — |

```sql
-- A executer par un admin, une fois les groupes crees dans la console Databricks (Settings > Identity and access)
GRANT USE CATALOG ON CATALOG workspace TO `data_analysts`;
GRANT USE SCHEMA, SELECT ON SCHEMA workspace.silver TO `data_analysts`;
GRANT USE SCHEMA, SELECT ON SCHEMA workspace.gold   TO `data_analysts`;

GRANT USE CATALOG ON CATALOG workspace TO `bi_users`;
GRANT USE SCHEMA, SELECT ON SCHEMA workspace.gold   TO `bi_users`;

GRANT ALL PRIVILEGES ON SCHEMA workspace.bronze TO `data_engineers`;
-- ... idem silver, gold, ops ; READ VOLUME, WRITE VOLUME ON VOLUME workspace.raw.landing

REVOKE SELECT ON SCHEMA workspace.silver FROM `bi_users`;   -- retirer un droit
```
⚠️ Ces commandes ne sont **pas exécutées** ici: les groupes n'existent pas dans ce workspace personnel.

### 4.3 Vue sécurisée pour les utilisateurs BI
Plutôt que d'exposer la table de faits complète, on publie une **vue** avec uniquement les colonnes utiles.

In [9]:
spark.sql(f"""
CREATE OR REPLACE VIEW {SCHEMA_GOLD}.v_trips_public
COMMENT 'Gold - Vue BI: courses sans colonnes techniques'
AS SELECT f.pickup_datetime, f.pickup_hour, pu.borough AS pickup_borough, pu.zone AS pickup_zone,
          do.borough AS dropoff_borough, do.zone AS dropoff_zone,
          f.trip_distance, f.trip_duration_min, f.total_amount, f.payment_type
   FROM {T_GOLD_FACT_TRIPS} f
   JOIN {T_GOLD_DIM_ZONE} pu ON f.pu_zone_key = pu.zone_key
   JOIN {T_GOLD_DIM_ZONE} do ON f.do_zone_key = do.zone_key
""")
spark.sql(f"SELECT * FROM {SCHEMA_GOLD}.v_trips_public LIMIT 5").show(truncate=False)

+-------------------+-----------+--------------+---------------------+---------------+------------------------+-------------+-----------------+------------+------------+
|pickup_datetime    |pickup_hour|pickup_borough|pickup_zone          |dropoff_borough|dropoff_zone            |trip_distance|trip_duration_min|total_amount|payment_type|
+-------------------+-----------+--------------+---------------------+---------------+------------------------+-------------+-----------------+------------+------------+
|2024-01-01 00:45:51|0          |Manhattan     |Upper West Side North|Manhattan      |Lincoln Square East     |0.7          |3.87             |13.80       |1           |
|2024-01-01 00:33:59|0          |Manhattan     |Clinton East         |Manhattan      |Lincoln Square East     |0.71         |5.9              |15.86       |1           |
|2024-01-01 00:45:10|0          |Manhattan     |Upper East Side North|Manhattan      |Washington Heights South|5.96         |28.4             |45.88  

## 5. Data Lineage

UC enregistre **automatiquement** la lignée des données (quelle table est construite à partir de quelle table, par quel notebook/job), au niveau **table** et **colonne**.

**Dans l'interface:** Catalog Explorer → `workspace.gold.fact_trips` → onglet **Lineage** → *See lineage graph*.

**Par SQL:** table système `system.access.table_lineage`.

In [10]:
try:
    lineage = spark.sql(f"""
        SELECT DISTINCT source_table_full_name AS source, target_table_full_name AS cible, entity_type
        FROM system.access.table_lineage
        WHERE target_table_catalog = '{CATALOG}'
          AND target_table_schema IN ('bronze', 'silver', 'gold', 'ops')
          AND source_table_full_name IS NOT NULL
        ORDER BY cible, source
    """)
    lineage.show(50, truncate=False)
except Exception as e:
    print("Tables systeme non accessibles ->", str(e).splitlines()[0][:200])
    print("Utiliser Catalog Explorer > Lineage.")

+-----------------------------+--------------------------------------+-----------+
|source                       |cible                                 |entity_type|
+-----------------------------+--------------------------------------+-----------+
|workspace.bronze.yellow_trips|workspace.ops.ingestion_log           |NULL       |
|workspace.bronze.taxi_zones  |workspace.silver.taxi_zones           |NULL       |
|workspace.bronze.yellow_trips|workspace.silver.yellow_trips         |NULL       |
|workspace.bronze.yellow_trips|workspace.silver.yellow_trips_rejected|NULL       |
+-----------------------------+--------------------------------------+-----------+



### Lignée attendue du projet
```
raw.landing (fichiers)
   └─► bronze.yellow_trips ─► silver.yellow_trips ─► gold.fact_trips ─┬─► gold.kpi_daily
                          └─► silver.yellow_trips_rejected            └─► gold.kpi_pickup_zone
   └─► bronze.taxi_zones ──► silver.taxi_zones ───► gold.dim_zone ────┘
                                                    gold.dim_date (depuis silver.yellow_trips)
```
*(La lignée système peut apparaître avec quelques minutes/heures de délai.)*

## Conclusion — Data Governance
- **Organisation** claire en 1 catalog / 5 schémas alignés sur l'architecture Medallion.
- **Documentation**: commentaires sur schémas, tables et colonnes clés + **tags** de classification.
- **Sécurité**: modèle de droits par groupe (moindre privilège) + **vue BI** qui masque les colonnes techniques.
- **Traçabilité**: lignée automatique table/colonne consultable dans l'UI et dans `system.access.table_lineage`.